# Classify Italian Brainrot characters

Train a small YOLO classifier on the [Italian Brainrot Images](https://www.kaggle.com/datasets/bubblepw/italian-brainrot-images) dataset, then watch it name a character.

Each photo's label is the folder it sits in:

- `ballerina_cappuccina`
- `bombardino_crocodilo`
- `cappuccino_assassino`
- `tralalero_tralala`
- `tung_tung_sahur`

Run the cells from top to bottom. The first training run downloads `yolo11n-cls.pt`. A GPU is used when one is available.

## Set up your computer

You only need a terminal and a web browser. Do these steps once, before you run any cells.

### 1. Open a terminal

- **Windows:** press the Windows key, type `PowerShell`, and open it.
- **Mac:** press Command+Space, type `Terminal`, and open it.
- **Linux:** open the Terminal app.

### 2. Install uv

uv downloads Python for you.

Mac or Linux:

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

Windows PowerShell:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

Close the terminal, open a new one, then check the install:

```bash
uv --version
```

### 3. Open this project folder

Use the folder you were given. Change the path if the folder is somewhere else on your computer.

```bash
cd italian_brainrot_image_classifier
```

### 4. Install the libraries

```bash
uv sync
```

The first run takes a few minutes. It creates a `.venv` folder and installs Python, PyTorch, and YOLO.

### 5. Open this notebook in your browser

```bash
uv run --with notebook jupyter notebook notebooks/01_classify_italian_brainrot.ipynb
```

The first time, that command also installs Jupyter. A browser tab then opens. Click a cell and press **Shift+Enter** to run it. Run the cells from top to bottom.

When you are finished, click the terminal and press **Ctrl+C**.

You also need a free [Kaggle](https://www.kaggle.com) account. The **Download the images** section shows where to put the API token.

## Download the images

Create an API token at [kaggle.com/settings](https://www.kaggle.com/settings) and save it as `~/.kaggle/kaggle.json`. On macOS or Linux, run `chmod 600 ~/.kaggle/kaggle.json`.

Running this cell again uses the copy already on your computer.

In [ ]:
import os

import kagglehub
import matplotlib.pyplot as plt
from PIL import Image
from ultralytics import YOLO

dataset_path = kagglehub.dataset_download("bubblepw/italian-brainrot-images")
image_folder = os.path.join(dataset_path, "brainrot_dataset")

class_names = [
    "ballerina_cappuccina",
    "bombardino_crocodilo",
    "cappuccino_assassino",
    "tralalero_tralala",
    "tung_tung_sahur",
]

print(image_folder)
for class_name in class_names:
    folder = os.path.join(image_folder, class_name)
    photo_count = 0
    for file_name in os.listdir(folder):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            photo_count = photo_count + 1
    print(class_name, photo_count)


def first_photo(class_name):
    """Return the path of the first image in a class folder."""
    folder = os.path.join(image_folder, class_name)
    for file_name in sorted(os.listdir(folder)):
        if file_name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            return os.path.join(folder, file_name)

## Look at one photo per character

In [ ]:
plt.figure(figsize=(12, 3))
for i, class_name in enumerate(class_names):
    photo = first_photo(class_name)
    plt.subplot(1, 5, i + 1)
    plt.imshow(Image.open(photo))
    plt.title(class_name.replace("_", " "), fontsize=9)
    plt.axis("off")
plt.show()

## Train

Five epochs is enough to see the model work while you watch. YOLO holds some photos back and scores itself on those.

In [ ]:
model = YOLO("yolo11n-cls.pt")
metrics = model.train(data=image_folder, epochs=5, imgsz=224, plots=False)
print(f"Got {metrics.top1:.0%} of the held-out photos right")

## Name one photo from each folder

The title is the model's answer, and the percent is how sure it is.

In [ ]:
plt.figure(figsize=(12, 3))
for i, class_name in enumerate(class_names):
    photo = first_photo(class_name)
    result = model(photo, verbose=False)[0]
    guess = result.names[result.probs.top1]
    confidence = float(result.probs.top1conf)

    plt.subplot(1, 5, i + 1)
    plt.imshow(Image.open(photo))
    plt.title(f"{guess.replace('_', ' ')}\n{confidence:.0%}", fontsize=9)
    plt.axis("off")
plt.show()